In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
laps = pd.read_csv('../data/processed/f1_race_laps.csv')
qual = pd.read_csv('../data/processed/f1_qualifying.csv')

print("Race laps:", laps.shape)
print("Qualifying:", qual.shape)
print("\nCircuits:", sorted(laps['GP'].unique()))
print("\nTeams:", sorted(laps['Team'].unique()))
print("\nYears:", sorted(laps['Year'].unique()))

In [4]:
# ── Cell 2: Basic lap time distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall lap time distribution
axes[0].hist(laps['LapTime'].dropna(), bins=60, color='#e10600', edgecolor='none', alpha=0.8)
axes[0].set_title('Lap time distribution (all circuits)')
axes[0].set_xlabel('Lap time (seconds)')
axes[0].set_ylabel('Count')

# Median lap time per circuit
circuit_median = (laps.groupby('GP')['LapTime']
                     .median()
                     .sort_values()
                     .reset_index())
axes[1].barh(circuit_median['GP'], circuit_median['LapTime'], color='#1e1e2e', edgecolor='none')
axes[1].set_title('Median lap time by circuit')
axes[1].set_xlabel('Lap time (seconds)')

plt.tight_layout()
plt.savefig('../data/processed/plot_lap_distribution.png', dpi=150)
plt.show()
print("Done")

In [5]:
# ── Cell 3: Tyre compound analysis ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lap time by compound (boxplot)
compound_order = ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']
compound_colors = {
    'SOFT': '#e10600', 'MEDIUM': '#ffd700',
    'HARD': '#f0f0f0', 'INTERMEDIATE': '#39b54a', 'WET': '#0067ff'
}
laps_clean = laps[laps['Compound'].isin(compound_order)]

sns.boxplot(
    data=laps_clean, x='Compound', y='LapTime',
    order=compound_order,
    palette=compound_colors,
    ax=axes[0], showfliers=False
)
axes[0].set_title('Lap time by tyre compound')
axes[0].set_xlabel('Compound')
axes[0].set_ylabel('Lap time (seconds)')

# Tyre degradation — lap time vs tyre age per compound
for compound in ['SOFT', 'MEDIUM', 'HARD']:
    subset = laps[laps['Compound'] == compound].dropna(subset=['TyreLife', 'LapTime'])
    grouped = subset.groupby('TyreLife')['LapTime'].median().reset_index()
    grouped = grouped[grouped['TyreLife'] <= 40]
    axes[1].plot(grouped['TyreLife'], grouped['LapTime'],
                 label=compound, color=compound_colors[compound], linewidth=2)

axes[1].set_title('Tyre degradation — lap time vs tyre age')
axes[1].set_xlabel('Tyre age (laps)')
axes[1].set_ylabel('Median lap time (seconds)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/plot_tyres.png', dpi=150)
plt.show()
print("Done")

In [8]:
# ── Cell 4: Team pace comparison ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Filter to dry laps only for fair comparison
dry_laps = laps[laps['Compound'].isin(['SOFT', 'MEDIUM', 'HARD'])]
dry_laps = dry_laps[dry_laps['IsAccurate'] == True]

# Median lap time per team per year
team_pace = (dry_laps.groupby(['Team', 'Year'])['LapTime']
             .median().reset_index())

# 2024 only — team ranking
pace_2024 = team_pace[team_pace['Year'] == 2024].sort_values('LapTime')
axes[0].barh(pace_2024['Team'], pace_2024['LapTime'], color='#e10600', edgecolor='none')
axes[0].set_title('Team median lap time — 2024')
axes[0].set_xlabel('Median lap time (seconds)')

# Team pace evolution across years
for team in dry_laps['Team'].unique():
    subset = team_pace[team_pace['Team'] == team]
    axes[1].plot(subset['Year'], subset['LapTime'], marker='o', label=team, linewidth=1.5)

axes[1].set_title('Team pace evolution 2022–2024')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Median lap time (seconds)')
axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig('../data/processed/plot_teams.png', dpi=150)
plt.show()
print("Done")

In [9]:
# ── Cell 5: Pit stop strategy ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Average pit stops per race per team
pit_laps = laps[laps['HasPitStop'] == True]
pit_counts = (pit_laps.groupby(['Team', 'Year', 'GP', 'Driver'])
              .size().reset_index(name='PitStops'))
avg_pits = pit_counts.groupby('Team')['PitStops'].mean().sort_values()

axes[0].barh(avg_pits.index, avg_pits.values, color='#1e1e2e', edgecolor='none')
axes[0].set_title('Avg pit stops per race per team')
axes[0].set_xlabel('Avg pit stops')
axes[0].axvline(avg_pits.mean(), color='#e10600', linestyle='--', label='Mean')
axes[0].legend()

# Pit stops per circuit
pit_circuit = (pit_counts.groupby('GP')['PitStops']
               .mean().sort_values(ascending=False))
axes[1].bar(range(len(pit_circuit)), pit_circuit.values, color='#e10600', edgecolor='none')
axes[1].set_xticks(range(len(pit_circuit)))
axes[1].set_xticklabels([gp.replace(' Grand Prix', '') for gp in pit_circuit.index],
                         rotation=90, fontsize=8)
axes[1].set_title('Avg pit stops by circuit')
axes[1].set_ylabel('Avg pit stops')

plt.tight_layout()
plt.savefig('../data/processed/plot_pitstops.png', dpi=150)
plt.show()
print("Done")

In [10]:
# ── Cell 6: Speed trap analysis (aero insight) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top speed per circuit (SpeedST = speed trap)
speed_circuit = (laps.groupby('GP')['SpeedST']
                 .median().sort_values(ascending=False)
                 .reset_index())
speed_circuit['GP'] = speed_circuit['GP'].str.replace(' Grand Prix', '')

axes[0].barh(speed_circuit['GP'], speed_circuit['SpeedST'],
             color='#e10600', edgecolor='none')
axes[0].set_title('Median top speed by circuit (speed trap)')
axes[0].set_xlabel('Speed (km/h)')

# Top speed per team (2024 only)
speed_team = (laps[laps['Year'] == 2024]
              .groupby('Team')['SpeedST']
              .median().sort_values(ascending=False)
              .reset_index())

axes[1].bar(range(len(speed_team)), speed_team['SpeedST'],
            color='#1e1e2e', edgecolor='none')
axes[1].set_xticks(range(len(speed_team)))
axes[1].set_xticklabels(speed_team['Team'], rotation=45, ha='right', fontsize=9)
axes[1].set_title('Median top speed by team — 2024')
axes[1].set_ylabel('Speed (km/h)')
axes[1].set_ylim(speed_team['SpeedST'].min() - 5, speed_team['SpeedST'].max() + 5)

plt.tight_layout()
plt.savefig('../data/processed/plot_speed.png', dpi=150)
plt.show()
print("Done")

In [11]:
# ── Cell 7: Qualifying vs race pace correlation ──
qual_clean = qual.dropna(subset=['Q1'])
qual_best = qual_clean.copy()
qual_best['BestQual'] = qual_best[['Q1','Q2','Q3']].min(axis=1)

race_pace = (laps[laps['IsAccurate'] == True]
             .groupby(['Driver','GP','Year'])['LapTime']
             .median().reset_index()
             .rename(columns={'LapTime': 'MedianRacePace'}))

merged = qual_best.merge(
    race_pace,
    left_on=['Abbreviation','GP','Year'],
    right_on=['Driver','GP','Year'],
    how='inner'
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(merged['BestQual'], merged['MedianRacePace'],
           alpha=0.4, color='#e10600', edgecolors='none', s=20)

# Trend line
z = np.polyfit(merged['BestQual'].dropna(),
               merged['MedianRacePace'].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(merged['BestQual'].min(), merged['BestQual'].max(), 100)
ax.plot(x_line, p(x_line), color='#1e1e2e', linewidth=2, label='Trend')

corr = merged['BestQual'].corr(merged['MedianRacePace'])
ax.set_title(f'Qualifying pace vs race pace (r = {corr:.2f})')
ax.set_xlabel('Best qualifying lap (seconds)')
ax.set_ylabel('Median race lap (seconds)')
ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/plot_qual_vs_race.png', dpi=150)
plt.show()
print(f"Correlation: {corr:.3f}")